# 02 — LightGBM benchmark

**Owners:** Saravana / Nebal
**Role:** high-performance nonlinear tree benchmark

Interview explanation: *“LightGBM learns nonlinear thresholds and interactions in
wide tabular data. It handles numerical missing values directly and does not require
scaling, making it both performant and efficient.”*


## Feature engineering for this approach

- Numerical quantities retain `NaN`; trees learn a missing-value branch.
- Low/medium-cardinality labels use stable training-only categorical levels.
- Rare training labels become `OTHER`; future unseen labels become `UNKNOWN`.
- High-cardinality labels and identifier-like numeric codes become frequency features.
- No standardization is used because split thresholds are invariant to scale.
- `scale_pos_weight` is calculated from the training partition only.


In [ ]:
from pathlib import Path
_install_root = Path.cwd().resolve()
for _candidate in [_install_root, *_install_root.parents]:
    if (_candidate / "requirements-training.txt").exists():
        _requirements = _candidate / "requirements-training.txt"
        break
else:
    raise FileNotFoundError("Open this notebook from inside the cloned repository")
%pip install -q -r {_requirements}


In [ ]:
from pathlib import Path
import gc, json, os, sys, time
import numpy as np
import pandas as pd

def locate_project_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / ".git").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Run this notebook from inside the cloned repository")

PROJECT_ROOT = locate_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DIR)


In [ ]:
required = [
    PROCESSED_DIR / "train.parquet",
    PROCESSED_DIR / "validation.parquet",
    PROCESSED_DIR / "test.parquet",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Run 00_shared_data_preparation.ipynb first. Missing: " + ", ".join(missing)
    )

train = pd.read_parquet(required[0])
validation = pd.read_parquet(required[1])
test = pd.read_parquet(required[2])

def stratified_debug_sample(frame, rows):
    if rows is None or rows >= len(frame):
        return frame
    return (
        frame.groupby("isFraud", group_keys=False)
        .apply(lambda group: group.sample(
            n=max(1, round(rows * len(group) / len(frame))),
            random_state=RANDOM_SEED,
        ), include_groups=True)
        .sort_values(["TransactionDT", "TransactionID"])
        .reset_index(drop=True)
    )

FAST_RUN = False  # Set True only to verify the notebook; never report these metrics.
if FAST_RUN:
    train = stratified_debug_sample(train, 60_000)
    validation = stratified_debug_sample(validation, 20_000)
    test = stratified_debug_sample(test, 20_000)

TARGET = "isFraud"
DROP_FROM_MODEL = ["isFraud", "TransactionID"]
X_train, y_train = train.drop(columns=DROP_FROM_MODEL), train[TARGET].astype("int8")
X_validation, y_validation = validation.drop(columns=DROP_FROM_MODEL), validation[TARGET].astype("int8")
X_test, y_test = test.drop(columns=DROP_FROM_MODEL), test[TARGET].astype("int8")

print("Train:", X_train.shape, "fraud rate:", f"{y_train.mean():.4%}")
print("Validation:", X_validation.shape, "fraud rate:", f"{y_validation.mean():.4%}")
print("Test:", X_test.shape, "fraud rate:", f"{y_test.mean():.4%}")


In [ ]:
MODEL_KEY = "lightgbm"


In [ ]:
from datetime import datetime, timezone
from src.fraud_pipeline.artifacts import build_manifest, package_versions, write_json
from src.fraud_pipeline.evaluation import evaluate_binary_classifier, select_operating_threshold

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = ARTIFACT_ROOT / MODEL_KEY / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("This run will be saved to:", RUN_DIR)


## Fit preprocessing on training only

The saved preprocessor contains category levels and frequency maps. Validation,
holdout, API requests, and batch predictions must use this same object.


In [ ]:
import joblib, lightgbm as lgb
from src.fraud_pipeline.preprocessing import LightGBMPreprocessor

preprocessor = LightGBMPreprocessor(low_cardinality_max=100, rare_min_count=20).fit(X_train)
X_train_model = preprocessor.transform(X_train)
X_validation_model = preprocessor.transform(X_validation)
print("Model matrix:", X_train_model.shape)
print("Native categorical columns:", len(preprocessor.categorical_features))


## Train with early stopping

Early stopping chooses the tree count using the validation period. Hyperparameters
should be changed only after recording this baseline.


In [ ]:
negative, positive = np.bincount(y_train)
scale_pos_weight = float(negative / positive)
model = lgb.LGBMClassifier(
    objective="binary",
    # Disable the default binary_logloss evaluation. With class weighting it can
    # worsen while fraud-ranking PR-AUC improves and select the wrong tree count.
    metric="None",
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=64,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
started = time.perf_counter()
model.fit(
    X_train_model,
    y_train,
    eval_set=[(X_validation_model, y_validation)],
    eval_metric="average_precision",
    categorical_feature=preprocessor.categorical_features,
    callbacks=[
        lgb.early_stopping(200, first_metric_only=True),
        lgb.log_evaluation(100),
    ],
)
training_seconds = time.perf_counter() - started
print("Best iteration:", model.best_iteration_)


## Validation threshold and final holdout evaluation


In [ ]:
validation_probability = model.predict_proba(X_validation_model, num_iteration=model.best_iteration_)[:, 1]
threshold_record = select_operating_threshold(y_validation, validation_probability, minimum_precision=0.10)
threshold = float(threshold_record["threshold"])
validation_metrics = evaluate_binary_classifier(y_validation, validation_probability, threshold)

del X_train_model
gc.collect()
X_test_model = preprocessor.transform(X_test)
started = time.perf_counter()
test_probability = model.predict_proba(X_test_model, num_iteration=model.best_iteration_)[:, 1]
prediction_seconds = time.perf_counter() - started
test_metrics = evaluate_binary_classifier(y_test, test_probability, threshold)
display(pd.DataFrame([validation_metrics, test_metrics], index=["validation", "test"])[["pr_auc", "roc_auc", "precision", "recall", "f1", "brier_score"]])


## Importance and native-format artifact

Gain importance shows which features reduced training loss most; it does not prove
causality. SHAP can be added after the baseline if memory permits.


In [ ]:
importance = pd.DataFrame({
    "feature": model.booster_.feature_name(),
    "gain": model.booster_.feature_importance(importance_type="gain"),
    "split": model.booster_.feature_importance(importance_type="split"),
}).sort_values("gain", ascending=False)
importance.head(100).to_csv(RUN_DIR / "feature_importance.csv", index=False)

model_path = RUN_DIR / "model.txt"
model.booster_.save_model(str(model_path), num_iteration=model.best_iteration_)
preprocessor_path = RUN_DIR / "preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path, compress=3)
pd.DataFrame({"TransactionID": validation["TransactionID"], "isFraud": y_validation, "probability": validation_probability}).to_parquet(RUN_DIR / "validation_predictions.parquet", index=False)
pd.DataFrame({"TransactionID": test["TransactionID"], "isFraud": y_test, "probability": test_probability}).to_parquet(RUN_DIR / "test_predictions.parquet", index=False)
write_json(RUN_DIR / "threshold.json", threshold_record)
write_json(RUN_DIR / "metrics.json", {"validation": validation_metrics, "test": test_metrics})
write_json(RUN_DIR / "feature_schema.json", {"model": MODEL_KEY, "groups": preprocessor.groups, "categorical_features": preprocessor.categorical_features})
write_json(RUN_DIR / "training_config.json", {
    "model": MODEL_KEY, "run_id": RUN_ID, "random_seed": RANDOM_SEED,
    "fast_run": FAST_RUN, "training_seconds": training_seconds,
    "test_prediction_seconds": prediction_seconds,
    "best_iteration": model.best_iteration_, "scale_pos_weight": scale_pos_weight,
    "parameters": model.get_params(),
    "versions": package_versions(["numpy", "pandas", "scikit-learn", "lightgbm", "joblib"]),
})


## Mandatory reload test


In [ ]:
loaded_preprocessor = joblib.load(preprocessor_path)
loaded_model = lgb.Booster(model_file=str(model_path))
sample_matrix = loaded_preprocessor.transform(X_validation.iloc[:5])
before = model.predict_proba(X_validation_model.iloc[:5], num_iteration=model.best_iteration_)[:, 1]
after = loaded_model.predict(sample_matrix)
np.testing.assert_allclose(before, after, rtol=1e-6, atol=1e-8)
write_json(RUN_DIR / "manifest.json", build_manifest(RUN_DIR))
print("Reload test passed:", after)
print("Artifact directory:", RUN_DIR)


In [ ]:
# Optional promotion step: upload this versioned run to a private Cloudflare R2 bucket.
# Create these as Lightning secrets/environment variables; never paste keys into a cell.
UPLOAD_TO_R2 = False

if UPLOAD_TO_R2:
    import boto3
    required_names = [
        "R2_ENDPOINT_URL", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_BUCKET_NAME"
    ]
    absent = [name for name in required_names if not os.getenv(name)]
    if absent:
        raise RuntimeError("Missing Lightning secrets: " + ", ".join(absent))
    client = boto3.client(
        "s3",
        endpoint_url=os.environ["R2_ENDPOINT_URL"],
        aws_access_key_id=os.environ["R2_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["R2_SECRET_ACCESS_KEY"],
        region_name="auto",
    )
    prefix = f"{MODEL_KEY}/{RUN_ID}"
    for local_path in RUN_DIR.rglob("*"):
        if local_path.is_file():
            key = f"{prefix}/{local_path.relative_to(RUN_DIR).as_posix()}"
            client.upload_file(str(local_path), os.environ["R2_BUCKET_NAME"], key)
    print(f"Uploaded to r2://{os.environ['R2_BUCKET_NAME']}/{prefix}/")
else:
    print("R2 upload skipped. Set UPLOAD_TO_R2=True after configuring Lightning secrets.")


## Interview checklist

Be ready to explain missing-value branches, native categoricals versus frequency
encoding, boosting, early stopping, `scale_pos_weight`, PR-AUC, and why scaling is
unnecessary for decision trees.
